# <font color="steelblue">Proyecto de clasificación — Riesgo en salud materna</font>

**Material desarrollado por los [equipos de trabajo de IA4LEGOS](https://ia4legos.umh.es/)**

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a>

No olvides hacer una copia si deseas utilizarlo.



## <font color="steelblue">Objetivos del proyecto</font>

A partir de **6 signos vitales** de gestantes (sistema IoT, Bangladesh, 2018–2020), construir, comparar y **desplegar** un clasificador que estime el **nivel de riesgo materno** en tres niveles **ordenados**: bajo, medio, alto. Lo que hace especial a este proyecto —y donde está su dificultad— es que el objetivo es **ordinal** y que hay solo **6 variables**, lo que exige cuidar la evaluación y la calidad del dato. Al terminar, debéis ser capaces de:

* Plantear una **clasificación ordinal** y evaluarla con **métricas conscientes del orden** (no todos los errores cuestan igual).
* Detectar y tratar problemas de **calidad** característicos de este dataset: **duplicados** y **valores atípicos**.
* Sacar partido a un EDA potente y a la **ingeniería de variables** cuando hay pocas características.
* **Comparar** modelos con metodología sólida, tratar el **desequilibrio**, **optimizar**, **combinar** si aporta, **interpretar** y **desplegar**.


## <font color="steelblue">El conjunto de datos</font>

### <font color="steelblue">Origen y estructura</font>

El conjunto procede del **UCI Machine Learning Repository** (id 863) y fue construido por Marzia Ahmed y Mohammod Kashem (2020) en el marco de un proyecto de salud materna. Los datos se recogieron en **hospitales, clínicas comunitarias y centros de salud materna de zonas rurales de Bangladés**, mediante un **sistema de monitorización basado en el internet de las cosas (IoT)**: sensores portátiles que registran de forma automática las constantes vitales de la gestante. El objetivo de fondo es la **reducción de la mortalidad materna**, uno de los Objetivos de Desarrollo Sostenible de la ONU, en un contexto donde el acceso al seguimiento médico especializado es limitado y una alerta temprana puede salvar vidas.

Contiene **1.014 registros** y **6 predictoras fisiológicas** más un objetivo ordinal de tres niveles. Todas las predictoras son **numéricas**, de modo que el preprocesado es mínimo; el peso del proyecto está, por tanto, en la **depuración** y en la **evaluación crítica**.

### <font color="steelblue">Diccionario de variables</font>

| Variable | Unidad | Descripción |
|---|---|---|
| `Age` | años | **Edad** de la gestante. |
| `SystolicBP` | mmHg | **Presión arterial sistólica** (la máxima, durante la contracción del corazón). Junto con la diastólica, es el marcador central del riesgo: la **hipertensión gestacional** y la **preeclampsia** son la principal causa de complicaciones graves en el embarazo. El umbral clínico habitual de hipertensión se sitúa en 140 mmHg. |
| `DiastolicBP` | mmHg | **Presión arterial diastólica** (la mínima, entre latidos). Su umbral de alarma se sitúa en torno a 90 mmHg. Está **fuertemente correlacionada** con la sistólica. |
| `BS` | mmol/L | **Glucosa en sangre**. Detecta la **diabetes gestacional**, que aumenta el riesgo de complicaciones para la madre y el feto. Los valores registrados sugieren que **no** se trata de glucemia en ayunas. Nota de unidades: en la práctica clínica anglosajona suele expresarse en mg/dL (1 mmol/L ≈ 18 mg/dL). |
| `BodyTemp` | **°F** | **Temperatura corporal**, en grados **Fahrenheit**. La fiebre puede indicar una **infección**, un factor de riesgo relevante durante la gestación. Como referencia: 98,6 °F = 37 °C, y el umbral de fiebre (100,4 °F) equivale a 38 °C. |
| `HeartRate` | lpm | **Frecuencia cardíaca** en reposo. Durante el embarazo aumenta de forma fisiológica; una taquicardia marcada puede señalar anemia, infección o descompensación. |

**Variable objetivo**

| Variable | Tipo | Valores | Descripción |
|---|---|---|---|
| `RiskLevel` | **Ordinal** | `low risk` < `mid risk` < `high risk` | Nivel de riesgo del embarazo. Fue asignado por **profesionales médicos** aplicando guías clínicas sobre las constantes anteriores; **no es una etiqueta automática**, pero tampoco un desenlace observado. |

**Distribución de clases.** El reparto es **moderadamente desequilibrado**: aproximadamente 406 registros de riesgo bajo (40 %), 336 de riesgo medio (33 %) y 272 de riesgo alto (27 %). No es un desequilibrio dramático, pero sí suficiente para desaconsejar la exactitud como métrica única.

> **Avisos de calidad (a investigar en la Fase 1):**
> * El dataset es conocido por contener **muchos registros duplicados**.
> * Hay **valores atípicos** evidentes (p. ej. `HeartRate` con valores imposibles como 7 bpm; `BodyTemp` en °F, casi constante en 98).
> * Las clases están **moderadamente desequilibradas**.

### <font color="steelblue">Advertencias metodológicas</font>

1. **Los duplicados son una fuente de fuga, no una simple redundancia.** Este es el punto crítico. Si una fila duplicada cae en entrenamiento y su gemela en test, el modelo **ya ha visto** exactamente esa observación: el resultado en test estará inflado y no medirá generalización alguna. La deduplicación debe hacerse **antes** de partir los datos, nunca después. Esto explica, muy probablemente, por qué la literatura reporta exactitudes del 97–98 % sobre un problema que, con solo seis constantes vitales, no debería permitirlas. Ahora bien, conviene pensar antes de borrar: con seis variables de valores discretos y redondeados, **dos gestantes distintas pueden tener legítimamente las mismas constantes**. Un duplicado no prueba un error de registro. La decisión —eliminar o conservar— debe **razonarse y documentarse**, y merece la pena comparar el rendimiento con y sin duplicados: la diferencia es, en sí misma, el resultado más instructivo del proyecto.

2. **Circularidad de la etiqueta.** `RiskLevel` no es un **desenlace observado** (no dice si la gestante sufrió complicaciones), sino la **clasificación que un profesional hizo aplicando umbrales de guía clínica a esas mismas seis variables**. Predecirlo equivale, en buena medida, a **reconstruir la regla de decisión del médico**, no a predecir la salud de la paciente. Es un ejercicio válido —automatizar un cribado tiene valor real en zonas sin especialistas—, pero exige llamarlo por su nombre: el modelo **replica un protocolo**, no descubre conocimiento clínico. Compruébalo examinando si los cortes que aprende un árbol coinciden con los umbrales clínicos (140 mmHg, 90 mmHg…).

3. **Valores fisiológicamente imposibles.** `HeartRate` contiene registros de 7 lpm, incompatibles con la vida. Y la variable `Age` abarca desde los **10 hasta los 70 años**, un rango implausible para una población de gestantes. No son *outliers* estadísticos que haya que tratar con delicadeza: son **errores de registro**. Documentad el criterio de eliminación y su efecto sobre el tamaño muestral.

4. **`BodyTemp` es casi constante.** La inmensa mayoría de los registros marcan 98 °F. Una variable con varianza casi nula aporta poquísima información y contribuirá muy poco al modelo… salvo en los pocos casos febriles, donde puede ser **decisiva**. Es un recordatorio de que la baja varianza no equivale a irrelevancia: no la descartéis mecánicamente. Considerad convertirla a °C, que es la escala en la que interpretaréis los resultados.

5. **La respuesta es ordinal.** `bajo < medio < alto` es una gradación de gravedad. Clasificar como «medio» un embarazo de riesgo «alto» es un error mucho menos grave que clasificarlo como «bajo». Acompañad la exactitud y el F1 de métricas **sensibles al orden** (QWK, MAE ordinal) y examinad la **matriz de confusión**: lo relevante es si los errores caen junto a la diagonal.

6. **El coste de los errores es radicalmente asimétrico.** Etiquetar como «bajo riesgo» a una gestante de riesgo alto puede tener consecuencias fatales; el error inverso solo genera una consulta adicional. La métrica que importa es el **recall de la clase `high risk`**, y el umbral de decisión debe fijarse en consecuencia.

7. **Correlación entre las presiones.** `SystolicBP` y `DiastolicBP` miden el mismo fenómeno y están fuertemente correlacionadas. No perjudica a los modelos basados en árboles, pero **reparte arbitrariamente su importancia** entre ambas: no concluyáis que una es prescindible porque su valor SHAP sea bajo.

8. **Validez externa limitada.** Los datos proceden de **una sola región rural de Bangladés** y no se documentan los criterios de inclusión ni de exclusión de las pacientes. Los umbrales aprendidos y las prevalencias observadas no son extrapolables a otra población, y el modelo resultante es un **ejercicio didáctico**, no una herramienta clínica.

## <font color="steelblue">Reglas del juego (buenas prácticas obligatorias)</font>

1. **El objetivo es ORDINAL.** Codificadlo respetando el orden (`low`=0 < `mid`=1 < `high`=2) y evaluad con métricas que **penalicen más los errores graves** (confundir *bajo* con *alto* es peor que *bajo* con *medio*).
2. **Duplicados primero.** Decidid qué hacer con ellos **antes** de partir: si los duplicados se reparten entre *train* y *test*, las métricas salen **infladas** (fuga por duplicación).
3. **Partición estratificada**; el *test* solo se toca al final.
4. **Preprocesado sin fuga:** escalado/remuestreo dentro de un **`Pipeline`**, ajustado solo con el *train* y rehecho en cada pliegue.
5. **El equilibrado solo en *train*** (material 11).
6. **Coste clínico:** no marcar a una gestante de **alto riesgo** es el error más grave; priorizad su **sensibilidad/recall**.
7. **Reproducibilidad y honestidad:** `random_state` fijado; reportad lo que no funcionó.

# <font color="steelblue">Fase 0 — Preparación del entorno y carga de datos</font>


In [ ]:
# !pip -q install ucimlrepo imbalanced-learn gradio optuna scikit-learn shap mord
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')
RNG = 42

In [ ]:
# Paso 1: descargar el dataset (corregido: usar maternal_health_risk, no chronic_kidney_disease)
from ucimlrepo import fetch_ucirepo
maternal_health_risk = fetch_ucirepo(id=863)
maternal_features = pd.DataFrame(maternal_health_risk.data.features)
maternal_target   = pd.DataFrame(maternal_health_risk.data.target)
maternal_health   = pd.concat([maternal_features, maternal_target], axis=1)
print(f"Dimensiones: {maternal_health.shape[0]:,} filas × {maternal_health.shape[1]} columnas")
maternal_health.head()

# <font color="steelblue">Fase 1 — Comprensión, calidad de datos y EDA</font>

Con solo 6 variables, podéis hacer un EDA muy visual y completo.

**Tareas obligatorias**
1. **Naturaleza ordinal.** Confirmad las 3 categorías de `RiskLevel` y su orden; calculad su distribución (¿desequilibrio?).
2. **Duplicados.** ¿Cuántas filas están repetidas? Decidid (y justificad) si las elimináis. *(Anotad la implicación para la partición.)*
3. **Atípicos / valores imposibles.** Revisad rangos: `HeartRate` (¿7 bpm?), `BodyTemp` (°F, casi constante), `BS` muy alto. Podéis apoyaros en el cuaderno de **Isolation Forest** del curso para detectarlos.
4. **EDA visual:** histogramas por variable, **boxplots por nivel de riesgo**, matriz de correlación y un **pairplot** coloreado por `RiskLevel`. ¿Qué variables separan mejor los niveles (probablemente `BS` y la presión)?
5. **Conclusión:** 3–4 hallazgos.

> **A responder:** ¿qué métricas usaréis dado que el objetivo es **ordinal**? (pista: *Kappa cuadrático*, MAE ordinal, F1-macro, y recall de *high risk*).

# <font color="steelblue">Fase 2 — Preprocesado: duplicados, atípicos, objetivo ordinal e ingeniería</font>

**2A. Limpieza (obligatorio)**
1. **Duplicados:** aplicad vuestra decisión **antes** de partir (justificadla).
2. **Atípicos:** tratad los valores imposibles (p. ej. `HeartRate` = 7) — corregir, eliminar o marcar; documentadlo.

**2B. Objetivo ordinal (clave)**
3. **Codificad `RiskLevel` respetando el orden:** `{'low risk':0, 'mid risk':1, 'high risk':2}`. Mantened a mano el mapeo inverso para interpretar.

**2C. Ingeniería de variables (recomendado, pocas columnas)**
4. Cread variables clínicas con sentido: **presión de pulso** (`SystolicBP − DiastolicBP`), **presión arterial media** (`DiastolicBP + (SystolicBP − DiastolicBP)/3`), indicadores de **fiebre** (`BodyTemp > 100.4 °F`) o **hiperglucemia** (umbral de `BS`). Evaluad si aportan.

**2D. Partición y `Pipeline`**
5. `X`/`y`, **partición estratificada**, y escalado (necesario para logística/SVM/kNN; los árboles no) dentro de `Pipeline`.

> **A responder:** ¿por qué los duplicados repartidos entre *train* y *test* inflan las métricas?

# <font color="steelblue">Fase 3 — Modelos base y comparación (con métricas de orden)</font>

**Tareas obligatorias**
1. Comparad **≥5 familias** del curso: **Regresión logística (multinomial)**, **kNN**, **SVM**, **Árbol**, **Random Forest**, **HistGradientBoosting** (o XGBoost/LightGBM/CatBoost), **Naive Bayes**.
2. **Validación cruzada repetida** estratificada (`RepeatedStratifiedKFold`); el dataset es pequeño, aprovechadlo.
3. **Métricas conscientes del orden** además de las habituales:
   * **F1-macro** / exactitud balanceada (tratan las 3 clases por igual),
   * **Kappa cuadrático** (`cohen_kappa_score(..., weights='quadratic')`): penaliza más los errores **lejanos** en la escala,
   * **MAE ordinal** (error medio sobre 0/1/2): cuánto se equivoca "de nivel".
4. **Tabla** comparativa y comentario.

> **Extra:** probad un modelo **ordinal** específico (paquete `mord`, regresión ordinal) y comparadlo con el enfoque multiclase estándar.

# <font color="steelblue">Fase 4 — Ponderación de muestras y equilibrado</font>

El desequilibrio es **moderado**, pero es obligatorio **medir** el efecto de tratarlo (material **11**), sobre los 2–3 mejores modelos:

1. **Sin tratamiento** (línea base).
2. **Sensible al coste:** `class_weight='balanced'` (o `sample_weight`). Pensad si interesa **penalizar especialmente** los errores que rebajan el riesgo de una gestante grave.
3. **Sobremuestreo:** **SMOTE** (variables numéricas; va dentro del `ImbPipeline`).
4. (Opcional) submuestreo/híbrido.

Reportad **F1-macro**, **QWK**, **recall de *high risk*** y exactitud balanceada, y comentad si aporta.

> **Sin fugas:** remuestreo dentro de `Pipeline` de *imbalanced-learn*, solo en *train*.

# <font color="steelblue">Fase 5 — Optimización de hiperparámetros</font>

1. Optimizad los **2–3 mejores** (modelo + equilibrado).
2. `GridSearchCV`/`RandomizedSearchCV`/**Optuna**, con CV estratificada y, preferiblemente, **`scoring` consciente del orden** (p. ej. el **Kappa cuadrático** como objetivo).
3. La búsqueda va **sobre el `Pipeline`** (prefijo `clf__`).
4. (Recomendado por el n pequeño) **CV anidada** para una estimación honesta.
5. Reportad mejores hiperparámetros y la mejora.

> El dataset es pequeño: una rejilla razonable es viable.

# <font color="steelblue">Fase 6 — Combinación de modelos</font>

1. Combinad los mejores con **`VotingClassifier`** (votación **blanda**) y/o **`StackingClassifier`**.
2. Comparad frente al **mejor individual** con las métricas de orden: ¿mejora el **QWK**/F1-macro?
3. **Combinad solo si aporta** mejora real (requisito: *si fuera necesario*).

# <font color="steelblue">Fase 7 — Evaluación, coste del error e interpretación</font>

**Tareas obligatorias** (el *test* se usa una sola vez)
1. Evaluad el **modelo final** en el *test*: **matriz de confusión** 3×3, `classification_report`, **F1-macro**, **QWK** y **MAE ordinal**.
2. **Análisis del error por gravedad.** Mirad específicamente las confusiones **bajo↔alto** (las más peligrosas) y el **recall de *high risk***. ¿Qué política de umbral/coste reduce los infra-diagnósticos de alto riesgo?
3. **Interpretabilidad:** `permutation_importance` y/o **SHAP** (multiclase). ¿Mandan `BS` y la presión arterial, como sugiere la clínica (preeclampsia, diabetes gestacional)?
4. **Atípicos (cierre):** comprobad que el tratamiento de outliers de la Fase 2 mejora (o no) la estabilidad del modelo.
5. **Discusión crítica:** origen único (Bangladesh, IoT), `BodyTemp` poco informativa, generalización a otras poblaciones.

# <font color="steelblue">Fase 8 — Despliegue del modelo</font>

1. **Persistencia:** guardad el **`Pipeline` completo** con `joblib`.
2. **Función de predicción:** `predecir_riesgo(Age, SystolicBP, DiastolicBP, BS, BodyTemp, HeartRate)` que devuelva el **nivel** (`low/mid/high`, usando el mapeo inverso) y las **probabilidades**.
3. **Interfaz interactiva:** app con **Gradio** (o `ipywidgets`) con los **6 signos vitales** como entradas y el nivel de riesgo como salida — coherente con el sistema **IoT** que originó los datos. En Colab genera un **enlace público** (incluidlo).
4. (Opcional, nota extra) **Streamlit**/**FastAPI**, o conectar a un flujo simulado de sensores.

> **Aviso clínico (obligatorio en la interfaz):** herramienta **educativa**, no sustituye la valoración de un profesional sanitario.

# <font color="steelblue">Pistas y errores típicos</font>

* **El objetivo está ordenado.** Confundir *bajo* con *alto* no es lo mismo que con *medio*: usa **Kappa cuadrático** y **MAE ordinal**, no solo *accuracy*.
* **Duplicados = fuga.** Si no los tratas antes de partir, el mismo registro puede estar en *train* y *test* y las métricas engañan.
* **Atípicos imposibles** (HeartRate de 7 bpm) distorsionan el escalado y los modelos basados en distancia (kNN, SVM): decídelos en la Fase 2.
* **Pocas variables → ingenia.** Presión de pulso y PAM pueden añadir señal; comprueba si ayudan.
* **Coste clínico asimétrico:** prioriza no perder a las gestantes de **alto riesgo** (su *recall*).
* **Despliegue:** guarda el **Pipeline entero** y respeta las columnas (incluidas las de ingeniería) en el orden de `X`.

# <font color="steelblue">Referencias</font>

* Ahmed, M., Kashem, M. A., Rahman, M. & Khatun, S. (2020). *Maternal Health Risk*. UCI ML Repository (id 863).
* Ahmed, M. & Kashem, M. A. (2020). *IoT Based Risk Level Prediction Model for Maternal Health Care in Bangladesh*. IEEE STI.
* *Deep hybrid model (ANN + Random Forest) for maternal health risk classification*. Frontiers in AI, 2023.
* WHO (2016). *Recommendations on antenatal care for a positive pregnancy experience*.
* Cuadernos del curso: *Equilibrando las muestras*, *Detección de anomalías (Isolation Forest)*, *Random Forest*, *Boosting*.